In [ ]:
import copy
import glob
import json
import librosa
import librosa.display
import random
import os
import sys
import torch

import IPython.display as ipd
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from collections import defaultdict
from scipy.io import wavfile
from scipy.spatial.distance import euclidean
from scipy import stats

sys.path.append("/orcd/data/jhm/001/om2/gelbanna/repos/projects/continuous-speech-recognition")

from front_end.cochlear_model import CochlearModel
from utils import load_yaml_config

#testing out cochlear model

In [ ]:
# Example instantiation
config = load_yaml_config("/orcd/data/jhm/001/om2/bjmedina/memory/utils/cochleagram_params.yaml")
config['frontend']['cochlear']['filter_params']['sr'] = 44100
model = CochlearModel(config.frontend.cochlear, device="cuda")


# Example forward pass
# x = [torch.randn(1, 1, 16000), torch.randn(1, 1, 8000), torch.randn(1, 1, 32000)]  # Example input tensor
# output = model(x[0].to("cuda"))
# print(output.shape)  # Example output shape 

In [ ]:
config

In [ ]:
def plot_torch_cochleagram(output):
    cochleagram_dB = output.squeeze(0).squeeze(0).detach().cpu().numpy()
    
    plt.figure(figsize=(10, 6))
    plt.imshow(
        cochleagram_dB,
        origin='lower',
        aspect='auto',
        # extent=[time_bins[0].item(), time_bins[-1].item(),
        #         freq_bins[0].item(), freq_bins[-1].item()],
        cmap='viridis'
    )
    plt.colorbar(label='Amplitude [dB]')
    plt.title("Cochleagram")
    plt.xlabel("Time [s]")
    plt.ylabel("Frequency [Hz]")
    plt.tight_layout()
    plt.show()
    
def plot_torch_corrs(output):
    corrs = output.squeeze(0).squeeze(0).detach().cpu().numpy()
    
    plt.figure(figsize=(10, 6))
    plt.matshow(
        corrs, vmin=0, vmax=1
    )
    plt.colorbar(label='Pearson Correlation')
    plt.title("Segments")
    plt.xlabel("Segments")
    plt.ylabel("Segments")
    plt.tight_layout()
    plt.show()
# plot_torch_cochleagram(output)

In [ ]:
def rms_normalize(signal, target_rms=0.1):
    """
    Normalize a signal to a target RMS level.
    
    Parameters:
        signal (numpy array): The input signal.
        target_rms (float): The desired RMS level (default is 0.1).
    
    Returns:
        numpy array: The RMS-normalized signal.
    """
    rms = np.sqrt(np.mean(signal**2))
    if rms == 0:
        return signal  # Avoid division by zero
    return signal * (target_rms / rms)

In [ ]:
# grabbing stationarity scores for sounds that are in the in the eval set of AudioSet (which is what we typically use for memory experiment)
stationarity_scores_path = "/orcd/data/jhm/001/om2/bjmedina/chexture_choolbox/assets/OVERLAPPED_0.5s_eval_sound_list_with_stationarity_score_no_speech_no_music_audioset_matlab_coch_rms0p02.csv"
stationarity_scores_ = pd.read_csv(stationarity_scores_path)

In [ ]:
len(set(stationarity_scores_.filepath.tolist()))

In [ ]:
# grabbing the textures that were selected to be in the memory 
texture_pairs_paths = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exemplar_0.5s_adjacent/texture_filenames.json"

with open(texture_pairs_paths) as f:
    texture_pairs = json.load(f)

In [ ]:
len(texture_pairs)

In [ ]:
ss_scores_to_texture = defaultdict(list)
times_to_texture     = defaultdict(list)

for texture_pair in texture_pairs:
    texture_id = "/".join(texture_pair['full_path'].split("/")[-2:])

    avg_ss_scores = stationarity_scores_[stationarity_scores_.filepath==texture_id].stationarity_score.tolist()
    times = stationarity_scores_[stationarity_scores_.filepath==texture_id].onset_time.tolist()
    
    ss_scores_to_texture[texture_id] = avg_ss_scores
    times_to_texture[texture_id]     = times

In [ ]:
# plot distribution of stationarity scores
avg_scores = []
for id in ss_scores_to_texture:
    avg_scores.extend(ss_scores_to_texture[id])

In [ ]:
plt.title("Stationarity Scores")
plt.axvline(x=np.mean(avg_scores), label=f"$\mu={np.mean(avg_scores)}$")
plt.hist(avg_scores, bins=100, alpha=0.5)
plt.legend()

avg_ss_score = np.mean(avg_scores)
std_ss_score = np.std(avg_scores)

In [ ]:
fps = stationarity_scores_[stationarity_scores_.stationarity_score < avg_ss_score].filepath.tolist(); 
fps = list(set(fps))

In [ ]:
len(fps)

In [ ]:
base_dir = "/om2/data/public/audioset/wavs/eval_segments_downloads/"
offset = 0.1

save_dir = f"/orcd/data/jhm/001/om2/bjmedina/data/texture_pairs/cochleagram_corrs-max_stationarity-offset{offset}/"

In [ ]:
import itertools

def abs_then_normalize(arr):
    """Min-max normalize an array to range [0,1]."""
    arr = np.abs(arr)  # Take absolute values first
    min_val, max_val = np.min(arr), np.max(arr)
    return (arr - min_val) / (max_val - min_val) if max_val > min_val else arr
    
def select_best_pair(cochleagram_corrs, time_avg_corrs, segment_times):
    """
    Selects the pair of unique audio clips that minimizes cochleagram correlation 
    while maximizing time-averaged cochleagram correlation.
    
    Args:
        cochleagram_corrs (torch.Tensor): 4x4 correlation matrix of cochleagrams.
        time_avg_corrs (torch.Tensor): 4x4 correlation matrix of time-averaged cochleagrams.

    Returns:
        tuple: The selected pair of clip indices.
    """
    num_clips = cochleagram_corrs.shape[0]

    # Ensure matrices are converted to NumPy for easier indexing
    cochleagram_corrs = cochleagram_corrs.cpu().numpy()
    time_avg_corrs = time_avg_corrs.cpu().numpy()

    norm_cochleagram_corrs = abs_then_normalize(cochleagram_corrs)
    norm_time_avg_corrs    = abs_then_normalize(time_avg_corrs)

    # Get all unique pairs of indices (excluding diagonal and duplicate pairs)
    pairs = list(itertools.combinations(range(num_clips), 2))

    best_pair = None
    best_score = -float('inf')  # We want to maximize this score

    valid_corrs = []
    valid_time_corrs = []

    segment_duration = 0.5
    best_time_corr = 0
    best_coch_corr = 1
    epsilon = 1e-5

    scores = []
    
    for i, j in pairs:

        if abs(segment_times[i] - segment_times[j]) >= segment_duration:
            coch_corr = norm_cochleagram_corrs[i, j]  # Lower is better
            time_corr = norm_time_avg_corrs[i, j]     # Higher is better

            

            valid_corrs.append(cochleagram_corrs[i,j])
            valid_time_corrs.append(time_avg_corrs[i,j])
    
            # Define a scoring function: maximize time correlation, minimize cochleagram correlation
            score = time_corr / (coch_corr+epsilon)  # Simple difference; can be weighted differently if needed
            scores.append(score)
    
            if score > best_score:
                best_score = score
                best_time_corr = time_avg_corrs[i,j] 
                best_coch_corr = cochleagram_corrs[i,j]
                best_pair = (i, j)

    return best_pair, best_time_corr, best_coch_corr, valid_corrs, valid_time_corrs, scores

def find_best_stationarity_match(times, stationarity_scores, target_stationarity):
    """
    Finds the time index with the closest stationarity score to the target.

    Args:
        times (list or array): List of time values.
        stationarity_scores (list or array): List of stationarity scores corresponding to times.
        target_stationarity (float): The desired stationarity score to match.

    Returns:
        tuple: (best_x, best_time, best_ss) where
            - best_x is the index of the closest stationarity score,
            - best_time is the corresponding time,
            - best_ss is the closest stationarity score.
    """
    min_mse = float('inf')  # Initialize with a very large number
    best_x = -1
    best_time = 0
    best_ss = 0

    for x, time in enumerate(times): 
        mse = (target_stationarity - stationarity_scores[x]) ** 2

        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            best_ss = stationarity_scores[x]

    return best_x, best_time, best_ss

def find_max_stationarity(times, stationarity_scores, target_stationarity):
    """
    Finds the time index with the closest stationarity score to the target.

    Args:
        times (list or array): List of time values.
        stationarity_scores (list or array): List of stationarity scores corresponding to times.
        target_stationarity (float): The desired stationarity score to match.

    Returns:
        tuple: (best_x, best_time, best_ss) where
            - best_x is the index of the closest stationarity score,
            - best_time is the corresponding time,
            - best_ss is the max stationarity score (least stationary sound).
    """
    max_mse = -float('inf')  # Initialize with a very large number
    best_x = -1
    best_time = 0
    best_ss = 0

    for x, time in enumerate(times): 
        mse = stationarity_scores[x]

        if mse > max_mse:
            best_x = x
            best_time = time
            min_mse = mse
            best_ss = stationarity_scores[x]

    return best_x, best_time, best_ss

def apply_linear_ramp(signal, sample_rate, ramp_duration_ms=5):
    """
    Applies a linear ramp (fade-in and fade-out) to a signal over a given duration.

    Args:
        signal (numpy.ndarray): Input audio signal (1D NumPy array).
        sample_rate (int): Sampling rate of the audio signal.
        ramp_duration_ms (float): Duration of the ramp in milliseconds (default: 5ms).

    Returns:
        numpy.ndarray: Signal with applied linear ramp.
    """
    num_samples = len(signal)
    ramp_samples = int((ramp_duration_ms / 1000) * sample_rate)  # Convert ms to samples

    if ramp_samples * 2 >= num_samples:
        raise ValueError("Ramp duration too long for the signal length!")

    # Create fade-in and fade-out ramps
    fade_in = np.linspace(0, 1, ramp_samples)
    fade_out = np.linspace(1, 0, ramp_samples)

    # Apply ramps to the signal
    signal[:ramp_samples] *= fade_in  # Apply fade-in
    signal[-ramp_samples:] *= fade_out  # Apply fade-out

    return signal

In [ ]:
pairs_ = []
best_time_corrs_ = []
best_corrs_ = []
corrs_ = []
time_corrs = []
best_sounds = []
ss_scores = []
target_stationarity = 0

target_sr = 22050*2

k = 0

best_pairs_dict = defaultdict(list)

for curr_sound in fps:

    # Compute cochleagram for each segment
    cochleagrams = []
    time_averaged_specs = []

    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]

    # if you have no stationarity scores for this sound, then move on to the next
    if len(stationarity) <= 0:
        #print("skipped")
        continue
    
    audio_path = base_dir + curr_sound
    
    data, samplerate = librosa.load(audio_path)
    
    if samplerate != target_sr:
        data = librosa.resample(data, orig_sr=samplerate, target_sr=target_sr)

    data = rms_normalize(data)
    
    # Parameters
    sr = samplerate
    segment_duration = 0.5  # seconds
    clip_duration = 2.0  # seconds
    num_samples_per_segment = int(segment_duration * target_sr)
    
    #print(best_ss)
    min_mse = 1e9
    best_x = -1
    best_time = 0
    best_ss = 0

    best_x, best_time, best_ss = find_max_stationarity(times, stationarity, target_stationarity)

    #print("best ss", best_ss)
    ss_scores.append(best_ss)
    
    sound = data[int(best_time*target_sr):int((best_time+2)*target_sr)]
    wav_data = sound
    
    
    # Generate all possible 0.5s segments
    segment_starts = np.arange(0, clip_duration - segment_duration, offset)  # Shift every 0.1s
    segments = []
    segment_times = []
    
    for start in segment_starts:
        start_sample = int(start * sr)
        end_sample = start_sample + num_samples_per_segment
        if end_sample <= len(wav_data):
            segments.append(rms_normalize(wav_data[start_sample:end_sample]))
            segment_times.append(start)  # Adjust to clip's time in full sound
    
    for segment in segments:
        segment_torch = torch.from_numpy(segment).unsqueeze(0).unsqueeze(0)
        S = model(segment_torch.to("cuda"))
    
        cochleagrams.append(S)
    
    #print(len(segments), len(cochleagrams))
    
    # calculating correlations between all cochleagrams
    cochleagram_tensor = torch.stack(cochleagrams).squeeze()
    coch_corr = torch.corrcoef(cochleagram_tensor.reshape(cochleagram_tensor.shape[0], -1))
    
    # calculating correlations between all time-averaged cochleagrams
    time_averaged_cochleagram = cochleagram_tensor.mean(dim=2)  
    time_averaged_coch_corr = torch.corrcoef(time_averaged_cochleagram.reshape(time_averaged_cochleagram.shape[0], -1))

    
    best_pair, best_time_corr, best_coch_corr, comparison_corrs, comparison_time_corrs, scores = select_best_pair(coch_corr, time_averaged_coch_corr, segment_times)
    best_time_corrs_.append(best_time_corr)
    best_corrs_.append(best_coch_corr)

    #if np.std(comparison_time_corrs) >= 0.1:


    # saving all values (this is keeping the diagonals, maybe we don't want that)
    print(f"Coch corrs: {comparison_corrs}")
    corrs_.extend(comparison_corrs)

    print(f"Time avg. Coch corrs: {comparison_time_corrs}")
    time_corrs.extend(comparison_time_corrs)
    
    pairs_.append(best_pair)
    
    # Extract the two most different segments
    segment_1 = segments[best_pair[0]]
    segment_2 = segments[best_pair[1]]

    best_sounds.append((curr_sound, segment_1, segment_2))
    
    # #print(best_pair[0], best_pair[1])
    
    #print(f"Most different segments: {segment_times[best_pair[0]]:.2f}s and {segment_times[best_pair[1]]:.2f}s")
    print(f"Most different segments: {segment_times[best_pair[0]]}s and {segment_times[best_pair[1]]}s with coch corr {best_coch_corr} and time avg corr {best_time_corr} ")
    
    norm_s1 = rms_normalize(segment_1)
    norm_s2 = rms_normalize(segment_2)
    
    display(ipd.Audio(norm_s1, rate=target_sr))
    display(ipd.Audio(norm_s2, rate=target_sr))

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    ID = curr_sound.split("/")[-1].split(".")[0]

    #print(save_dir+f"{ID}_0.wav")

    wavfile.write(save_dir+f"{ID}_0.wav", target_sr, norm_s1.astype(np.float32))
    wavfile.write(save_dir+f"{ID}_1.wav", target_sr, norm_s2.astype(np.float32))
    

    k += 1

    plt.title(best_pair)
    plt.scatter(comparison_corrs, scores, alpha=0.5, c='r', label=f"$\mu={np.mean(comparison_corrs)}$, $\sigma={np.std(comparison_corrs)}$")
    plt.scatter(comparison_time_corrs, scores, alpha=0.5, c='b', label=f"$\mu={np.mean(comparison_time_corrs)}$, $\sigma={np.std(comparison_time_corrs)}$")
    plt.legend()
    plt.show()
    input()
    plt.clf()

    clear_output(wait=False)


In [ ]:
# corrs with means at one end of spectrum and small standard deviations tend to sound very similar to one another
# sigma < 0.02
# if sigma > 0.1 then that could work. 

In [ ]:
segment_starts, len(segments[-1])

In [ ]:
plt.hist(ss_scores, bins=len(ss_scores))
np.sum(np.array(ss_scores)>-0.4)

In [ ]:
# plt.hist(time_corrs, bins=1000, alpha=0.5, density=True)
# plt.hist(best_time_corrs_, bins=1000, alpha=0.5, density=True
print("Average correlation between time-averaged cochleagrams", np.mean(time_corrs))

In [ ]:
# plt.hist(corrs_, bins=1000, alpha=0.5, density=True)
# plt.hist(best_corrs_, bins=1000, alpha=0.5, density=True
print("Average correlation between cochleagrams", np.mean(corrs_))

In [ ]:
print(len(best_time_corrs_))

In [ ]:
stat_cutoff = -0.499
save_dir_filtered = f"/orcd/data/jhm/001/om2/bjmedina/data/texture_pairs/cochleagram_corrs-max_stationarity_filtered_greaterthan{stat_cutoff}_offset-{offset}/"

In [ ]:
# Get indices where A > a and B < b

mean_corr = np.mean(corrs_)
std_corr  = 0#np.std(corrs_)
mean_time_corr = np.mean(time_corrs)
std_time_corr  = 0#np.std(time_corrs)
indices = [i for i in range(len(best_corrs_)) if (best_corrs_[i] <= mean_corr and best_time_corrs_[i] >= mean_time_corr and ss_scores[i] >= stat_cutoff)]

print(len(indices))

In [ ]:
filtered_best = [best_sounds[best_idx] for best_idx in indices]

In [ ]:
best_sounds[0][1]

In [ ]:
# norm_s1 = rms_normalize(segment_1)
# norm_s2 = rms_normalize(segment_2)
chosen_idx = 14
# # display(ipd.Audio(norm_s1, rate=samplerate))
# # display(ipd.Audio(norm_s2, rate=samplerate))
# display(ipd.Audio(filtered_best[chosen_idx][1], rate=target_sr))
# display(ipd.Audio(filtered_best[chosen_idx][2], rate=target_sr))

for ID, pair in enumerate(filtered_best):
    
    segment_1 = pair[1]
    segment_2 = pair[2]    
    #print(f"Most different segments: {segment_times[best_pair[0]]:.2f}s and {segment_times[best_pair[1]]:.2f}s")
    # #print(f"Most different segments: {segment_times[best_pair[0]]}s and {segment_times[best_pair[1]]}s with distance {max_distance}")
    #display(ipd.Audio(norm_s1, rate=target_sr))
    #display(ipd.Audio(norm_s2, rate=target_sr))

    if not os.path.exists(save_dir_filtered):
        os.makedirs(save_dir_filtered)
    
    #print(save_dir+f"{ID}_0.wav")

    wavfile.write(save_dir_filtered+f"mem{ID}_0.wav", target_sr, segment_1.astype(np.float32))
    wavfile.write(save_dir_filtered+f"mem{ID}_1.wav", target_sr, segment_2.astype(np.float32))


In [ ]:
offset 

In [ ]:

offset = 0.5
save_dir = f"/orcd/data/jhm/001/om2/bjmedina/data/texture_pairs/cochleagram_corrs-full_clip-offset{offset}/"
pairs_ = []
best_time_corrs_ = []
best_corrs_ = []
corrs_ = []
time_corrs = []
best_sounds = []
ss_scores = []
all_scores = []
target_stationarity = 0

target_sr = 22050*2

clip_info = {}

k = 0

best_pairs_dict = defaultdict(list)

for curr_sound in fps:

    to_save = {}

    # Compute cochleagram for each segment
    cochleagrams = []
    time_averaged_specs = []

    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    audio_path = base_dir + curr_sound
    
    data, samplerate = librosa.load(audio_path)
    
    if samplerate != target_sr:
        data = librosa.resample(data, orig_sr=samplerate, target_sr=target_sr)

    data = rms_normalize(data)

    wav_data = data
    
    # Generate all possible 0.5s segments
    sr = samplerate
    clip_duration = 10.0
    segment_duration = 0.5
    segment_starts = np.arange(0, clip_duration - segment_duration, offset)  # Shift every 0.1s
    num_samples_per_segment = int(segment_duration * target_sr)
    segments = []
    segment_times = []
    
    for start in segment_starts:
        start_sample = int(start * sr)
        end_sample = start_sample + num_samples_per_segment
        if end_sample <= len(wav_data):
            segments.append(rms_normalize(wav_data[start_sample:end_sample]))
            segment_times.append(start)  # Adjust to clip's time in full sound
    
    for segment in segments:
        segment_torch = torch.from_numpy(segment).unsqueeze(0).unsqueeze(0)
        S = model(segment_torch.to("cuda"))
    
        cochleagrams.append(S)
    
    #print(len(segments), len(cochleagrams))
    
    # calculating correlations between all cochleagrams
    cochleagram_tensor = torch.stack(cochleagrams).squeeze()
    coch_corr = torch.corrcoef(cochleagram_tensor.reshape(cochleagram_tensor.shape[0], -1))
    
    # calculating correlations between all time-averaged cochleagrams
    time_averaged_cochleagram = cochleagram_tensor.mean(dim=2)  
    time_averaged_coch_corr = torch.corrcoef(time_averaged_cochleagram.reshape(time_averaged_cochleagram.shape[0], -1))

    
    best_pair, best_time_corr, best_coch_corr, comparison_corrs, comparison_time_corrs, scores = select_best_pair(coch_corr, time_averaged_coch_corr, segment_times)
    best_time_corrs_.append(best_time_corr)
    best_corrs_.append(best_coch_corr)
    
    # saving all values (this is keeping the diagonals, maybe we don't want that)
    #print(f"Coch corrs: {comparison_corrs}")
    corrs_.extend(comparison_corrs)

    #print(f"Time avg. Coch corrs: {comparison_time_corrs}")
    time_corrs.extend(comparison_time_corrs)

    all_scores.extend(scores)


    to_save["best_time_corr"] = best_time_corr
    to_save["best_coch_corr"] = best_coch_corr
    to_save["clip_1_onset"]   = segment_times[best_pair[0]]
    to_save["clip_2_onset"]   = segment_times[best_pair[1]]
    to_save["best_pair"]      = best_pair
    to_save["segment_times"]  = segment_times
    to_save["all_coch_corrs"] = comparison_corrs
    to_save["all_time_coch_corrs"] = comparison_time_corrs
    to_save["scores"] = scores

    clip_info[curr_sound] = to_save
    
    
    
    # pairs_.append(best_pair)
    
    # # Extract the two most different segments
    # segment_1 = segments[best_pair[0]]
    # segment_2 = segments[best_pair[1]]

    # best_sounds.append((curr_sound, segment_1, segment_2))
    
    # # #print(best_pair[0], best_pair[1])
    
    # #print(f"Most different segments: {segment_times[best_pair[0]]:.2f}s and {segment_times[best_pair[1]]:.2f}s")
    # print(f"Most different segments: {segment_times[best_pair[0]]}s and {segment_times[best_pair[1]]}s with coch corr {best_coch_corr} and time avg corr {best_time_corr} ")
    
    # norm_s1 = rms_normalize(segment_1)
    # norm_s2 = rms_normalize(segment_2)
    
    # display(ipd.Audio(norm_s1, rate=target_sr))
    # display(ipd.Audio(norm_s2, rate=target_sr))
    #     os.makedirs(save_dir)
    
    # ID = curr_sound.split("/")[-1].split(".")[0]

    # #print(save_dir+f"{ID}_0.wav")

    # wavfile.write(save_dir+f"{ID}_0.wav", target_sr, norm_s1.astype(np.float32))
    # wavfile.write(save_dir+f"{ID}_1.wav", target_sr, norm_s2.astype(np.float32))
    

    # k += 1

    # print("score", scores)
    # plt.title(f"{best_time_corr}, {best_coch_corr}")
    # plt.scatter(comparison_corrs, scores, alpha=0.5, c='r', label=f"$\mu={np.mean(comparison_corrs)}$, $\sigma={np.std(comparison_corrs)}$")
    # plt.scatter(comparison_time_corrs, scores, alpha=0.5, c='b', label=f"$\mu={np.mean(comparison_time_corrs)}$, $\sigma={np.std(comparison_time_corrs)}$")
    # plt.legend()
    # plt.show()
    # input()
    # plt.clf()

    # clear_output(wait=False)


In [ ]:
print(len(clip_info))


all_coch_corrs = []
all_time_coch_corrs = []
all_scores = []

best_coch_corrs = []
best_time_coch_corrs = []
best_scores = []

for key in clip_info:
    curr_pair = clip_info[key]
    all_coch_corrs.extend(curr_pair["all_coch_corrs"])
    all_time_coch_corrs.extend(curr_pair["all_time_coch_corrs"])
    best_coch_corrs.append(curr_pair['best_coch_corr'])
    best_time_coch_corrs.append(curr_pair['best_time_corr'])

    all_scores.extend(curr_pair["scores"])
    best_scores.append(np.max(curr_pair["scores"]))

print(len(all_coch_corrs), len(all_time_coch_corrs))
print(len(best_coch_corrs), len(best_time_coch_corrs))
print(len(segment_starts))


x = plt.hist(all_coch_corrs, bins=100, color='r', alpha=0.5, density=True, label="All Coch Correlations")
y = plt.hist(best_coch_corrs, bins=100, color='m', alpha=0.5, density=True, label="Best Coch Correlations")


x = plt.hist(all_time_coch_corrs, bins=100, color='b', alpha=0.5, density=True, label="All Time-averaged Coch Correlations")
y = plt.hist(best_time_coch_corrs, bins=100, color='g', alpha=0.5, density=True, label="Best Time-averaged Coch Correlations")

plt.legend()
plt.title("Distribution of pairwise cochleagram correlations")
plt.ylabel("Density")
plt.xlabel("Pairwise Correlation")

In [ ]:
sound_idx = 1248
curr_pair = clip_info[list(clip_info.keys())[sound_idx]]
print(curr_pair.keys())

# graph is useful because i would EXCEPT to see 
## scores increase as pairwise time-average cochleagram correlations increase
## scores increase as pairwise cochleagram correlations decrease

plt.title(f"Scores")
plt.hist(curr_pair["scores"], label="All pairwise scores", bins=len(curr_pair["scores"]), alpha=0.5, color='r')
plt.axvline(np.max(curr_pair['scores']), color='b', alpha=0.5)

#plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
plt.ylabel("Density")
plt.xlabel("Score")


plt.legend()

In [ ]:
sound_idx = 1248
curr_pair = clip_info[list(clip_info.keys())[sound_idx]]
print(curr_pair.keys())

# graph is useful because i would EXCEPT to see 
## scores increase as pairwise time-average cochleagram correlations increase
## scores increase as pairwise cochleagram correlations decrease

plt.title(f"Best time-average coch corr {curr_pair['best_time_corr']:.2f} \n best coch corr {curr_pair['best_coch_corr']:.2f}")
plt.scatter(curr_pair["all_coch_corrs"], curr_pair["scores"], label="All pairwise cochleagram correlations", alpha=0.5, color='r')
plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
plt.ylabel("Score")
plt.xlabel("Pairwise-Correlation")
plt.xlim([-0.1, 1.1])

plt.axvline(curr_pair['best_time_corr'], color='b', alpha=0.5)
plt.axvline(curr_pair['best_coch_corr'], color='r', alpha=0.5)

plt.legend()

In [ ]:
best_coch_corr = []
best_time_corr = []

thres_coch = np.mean(all_coch_corrs) - np.std(all_coch_corrs)/2
thres_time = np.mean(all_time_coch_corrs) + np.std(all_time_coch_corrs)/2

print(thres_coch, thres_time)

filtered_clip_info = {}

for clip_id in clip_info.keys():
    clip_coch_corr = clip_info[clip_id]["best_coch_corr"]
    clip_time_corr = clip_info[clip_id]["best_time_corr"]

    if clip_coch_corr <= thres_coch and clip_time_corr >= thres_time:
    
        best_time_corr.append(clip_info[clip_id]["best_time_corr"])
        best_coch_corr.append(clip_info[clip_id]["best_coch_corr"])

        filtered_clip_info[clip_id] = copy.deepcopy(clip_info[clip_id])

print(len(filtered_clip_info))

In [ ]:
target_sr = 22050*2
segment_duration = 0.5
num_samples_per_segment = int(segment_duration * target_sr)
k = 0

for curr_sound in filtered_clip_info.keys():

    curr_pair = filtered_clip_info[curr_sound]


    
    # get audiopath
    audio_path = base_dir + curr_sound

    # grab the samples
    data, samplerate = librosa.load(audio_path)

    # 
    if samplerate != target_sr:
        data = librosa.resample(data, orig_sr=samplerate, target_sr=target_sr)

    # rms normalize
    data = rms_normalize(data)

    # first segment start time
    start_sample_1 = int(filtered_clip_info[curr_sound]['clip_1_onset']*target_sr)
    end_sample_1   = start_sample_1 + num_samples_per_segment
    segment_1 = rms_normalize(data[start_sample_1:end_sample_1])
    segment_1 = apply_linear_ramp(segment_1, target_sr)


    # first segment start time
    start_sample_2 = int(filtered_clip_info[curr_sound]['clip_2_onset']*target_sr)
    end_sample_2   = start_sample_2 + num_samples_per_segment 
    segment_2 = rms_normalize(data[start_sample_2:end_sample_2])
    segment_2 = apply_linear_ramp(segment_2, target_sr)

    display(ipd.Audio(segment_1, rate=target_sr))
    display(ipd.Audio(segment_2, rate=target_sr))

    ID = curr_sound.split("/")[-1].split(".")[0]

    # #print(save_dir+f"{ID}_0.wav")

    wavfile.write(save_dir+f"{ID}_0.wav", target_sr, segment_1.astype(np.float32))
    wavfile.write(save_dir+f"{ID}_1.wav", target_sr, segment_2.astype(np.float32))


    plt.title(f"Best time-average coch corr {curr_pair['best_time_corr']:.2f} \n best coch corr {curr_pair['best_coch_corr']:.2f}")
    plt.scatter(curr_pair["all_coch_corrs"], curr_pair["scores"], label="All pairwise cbochleagram correlations", alpha=0.5, color='r')
    plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
    plt.ylabel("Score")
    plt.xlabel("Pairwise-Correlation")
    plt.xlim([-0.1, 1.1])
    
    plt.axvline(curr_pair['best_time_corr'], color='b', alpha=0.5)
    plt.axvline(curr_pair['best_coch_corr'], color='r', alpha=0.5)
    
    plt.legend()
    plt.savefig(save_dir+f"{ID}.png")
    plt.clf()


    plt.title(f"Best time-average coch corr {curr_pair['best_time_corr']:.2f} \n best coch corr {curr_pair['best_coch_corr']:.2f}")
    plt.scatter(curr_pair["all_coch_corrs"], curr_pair["scores"], label="All pairwise cochleagram correlations", alpha=0.5, color='r')
    plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
    plt.ylabel("Score")
    plt.xlabel("Pairwise-Correlation")
    plt.xlim([-0.1, 1.1])
    
    plt.axvline(curr_pair['best_time_corr'], color='b', alpha=0.5)
    plt.axvline(curr_pair['best_coch_corr'], color='r', alpha=0.5)
    
    plt.legend()
    plt.savefig(save_dir+f"{ID}.png")
    plt.clf()

    plt.title(f"Scores")
    plt.hist(curr_pair["scores"], label="All pairwise scores", bins=len(curr_pair["scores"]), alpha=0.5, color='r')
    plt.axvline(np.max(curr_pair['scores']), color='b', alpha=0.5)
    
    #plt.scatter(curr_pair["all_time_coch_corrs"], curr_pair["scores"], label="All pairwise time-averaged cochleagram correlations", alpha=0.5, color='b')
    plt.ylabel("Density")
    plt.xlabel("Score")
    
    plt.savefig(save_dir+f"{ID}_scores.png")

    plt.legend()

    plt.clf()
    # k += 1
    
    #input()
    clear_output(wait=False)

In [ ]:
save_dir, curr_sound.split("/")[-1].split(".")[0]

In [ ]:
len(glob.glob(f"/orcd/data/jhm/001/om2/bjmedina/data/texture_pairs/cochleagram_corrs-full_clip-offset{offset}/*wav"))//2

In [ ]:
save_dir = "/orcd/data/jhm/001/om2/bjmedina/data/texture_pairs/spectral_contrast-stationarity-score{}/"

In [ ]:
# index of memory stimulus
k = 0
target_stationarity = -0.5


max_distances = []
# loop through all sounds in the eval set
for fp in fps:    
    stationarity, times = [], []

    #print(fp)
    curr_sound = fp
    audio_path = base_dir + curr_sound
    ipd.Audio(audio_path)
    
    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    samplerate, data = wavfile.read(audio_path)
    try:
        data = data[:, 0].astype(np.float32)
    except:
        data = data.astype(np.float32)
    ipd.Audio(data, rate=samplerate)

    if len(stationarity) == 0:
        continue

    # Ideally, i think we should take all the sounds that have segments adjacent to each other that are stationary in time
    numbers = times
    #print(numbers)
    pairs = [(i, i+1) for i in range(len(numbers)-1) if abs(numbers[i] - numbers[i+1]) <= 1.0]
    
    #print(pairs)
    
    #TODO:
    # look for least stationary segment in clip, 
    # now that the whole clip is not stationary, find the two most different sections in that whole clip. 
    
    stationarity_scores = ss_scores_to_texture[curr_sound]
    
    min_mse = 1e9
    best_x = -1
    best_time = 0
    
    for x, time in enumerate(numbers): 
    
        mse = (target_stationarity - stationarity_scores[x]) ** 2
    
        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            
    #print(best_x, best_time)
    sound = data[int(best_time*samplerate):int((best_time+2)*samplerate)]
    
    wav_data = sound

    # grabbing all non-overlapping 0.5s segments from the 2s segment that has the stationarity closest to -0.6
    sr = samplerate  # Assume a common sample rate
    segment_duration = 0.5  # seconds
    
    # Assume `wav_data` contains the raw audio data
    # Assume `sr` is the sample rate of `wav_data`
    num_samples_per_segment = int(segment_duration * sr)
    num_segments = len(wav_data) // num_samples_per_segment
    
    # Split audio into non-overlapping 0.5s segments
    segments = [
        wav_data[i * num_samples_per_segment : (i + 1) * num_samples_per_segment]
        for i in range(num_segments)
    ]
    
    #print(num_segments)
    segment_onsets = [i*segment_duration for i in range(num_segments)]
    
    segment_onsets_relative_to_beginning = [(best_time+i*segment_duration) for i in range(num_segments)]
    #print(segment_onsets_relative_to_beginning)


    # Compute spectral contrast for each segment
    spectral_contrast_features = []
    for segment in segments: 
        S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
        contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
        spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time
    
    # Compute Euclidean distances between all pairs
    num_segments = len(spectral_contrast_features)
    max_distance = -1
    best_pair = (0, 1)
    
    for i in range(num_segments):
        for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
            dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
            if dist > max_distance:
                max_distance = dist
                best_pair = (i, j)

    max_distances.append(max_distance)
    
    # # Extract the two most different segments
    # segment_1 = segments[best_pair[0]]
    # segment_2 = segments[best_pair[1]]
    
    # # Print results
    # #print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")
    
    # #print("best segments")

    # norm_s1 = rms_normalize(segment_1)
    # norm_s2 = rms_normalize(segment_2)

    # #display(ipd.Audio(norm_s1, rate=samplerate))
    # #display(ipd.Audio(norm_s2, rate=samplerate))

    # if not os.path.exists(save_dir.format(target_stationarity)):
    #     os.makedirs(save_dir.format(target_stationarity))

    # wavfile.write(save_dir.format(target_stationarity)+f"mem_stim_{k}_0.wav", samplerate, norm_s1.astype(np.float32))
    # wavfile.write(save_dir.format(target_stationarity)+f"mem_stim_{k}_1.wav", samplerate, norm_s2.astype(np.float32))

    k+=1

    #print(f"Sound {stationarity_scores[best_x]}:")
    #input()

In [ ]:
x = plt.hist(max_distances, bins=len(max_distances))
plt.axvline(x=np.mean(max_distances), c='r')

the_bar = np.mean(max_distances) + np.std(max_distances)

In [ ]:
# index of memory stimulus
k = 0

max_distances = []
# loop through all sounds in the eval set
for fp in fps:    
    stationarity, times = [], []

    #print(fp)
    curr_sound = fp
    audio_path = base_dir + curr_sound
    ipd.Audio(audio_path)
    
    stationarity, times = ss_scores_to_texture[curr_sound], times_to_texture[curr_sound]
    
    samplerate, data = wavfile.read(audio_path)
    try:
        data = data[:, 0].astype(np.float32)
    except:
        data = data.astype(np.float32)
    ipd.Audio(data, rate=samplerate)

    if len(stationarity) == 0:
        continue

    # Ideally, i think we should take all the sounds that have segments adjacent to each other that are stationary in time
    numbers = times
    #print(numbers)
    pairs = [(i, i+1) for i in range(len(numbers)-1) if abs(numbers[i] - numbers[i+1]) <= 1.0]
    
    #print(pairs)
    
    #TODO:
    # look for least stationary segment in clip, 
    # now that the whole clip is not stationary, find the two most different sections in that whole clip. 
    
    stationarity_scores = ss_scores_to_texture[curr_sound]
    
    min_mse = 1e9
    best_x = -1
    best_time = 0
    
    for x, time in enumerate(numbers): 
    
        mse = (target_stationarity - stationarity_scores[x]) ** 2
    
        if mse < min_mse:
            best_x = x
            best_time = time
            min_mse = mse
            
    #print(best_x, best_time)
    sound = data[int(best_time*samplerate):int((best_time+2)*samplerate)]
    
    wav_data = sound

    # grabbing all non-overlapping 0.5s segments from the 2s segment that has the stationarity closest to -0.6
    sr = samplerate  # Assume a common sample rate
    segment_duration = 0.5  # seconds
    
    # Assume `wav_data` contains the raw audio data
    # Assume `sr` is the sample rate of `wav_data`
    num_samples_per_segment = int(segment_duration * sr)
    num_segments = len(wav_data) // num_samples_per_segment
    
    # Split audio into non-overlapping 0.5s segments
    segments = [
        wav_data[i * num_samples_per_segment : (i + 1) * num_samples_per_segment]
        for i in range(num_segments)
    ]
    
    #print(num_segments)
    segment_onsets = [i*segment_duration for i in range(num_segments)]
    
    segment_onsets_relative_to_beginning = [(best_time+i*segment_duration) for i in range(num_segments)]
    #print(segment_onsets_relative_to_beginning)


    # Compute spectral contrast for each segment
    spectral_contrast_features = []
    for segment in segments: 
        S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
        contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
        spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time
    
    # Compute Euclidean distances between all pairs
    num_segments = len(spectral_contrast_features)
    max_distance = -1
    best_pair = (0, 1)
    
    for i in range(num_segments):
        for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
            dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
            if dist > max_distance:
                max_distance = dist
                best_pair = (i, j)

    #max_distances.append(max_distance)

    if max_distance <= the_bar:
        continue
    
    # Extract the two most different segments
    segment_1 = segments[best_pair[0]]
    segment_2 = segments[best_pair[1]]
    
    # Print results
    #print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")
    
    #print("best segments")

    norm_s1 = rms_normalize(segment_1)
    norm_s2 = rms_normalize(segment_2)

    #display(ipd.Audio(norm_s1, rate=samplerate))
    #display(ipd.Audio(norm_s2, rate=samplerate))

    if not os.path.exists(save_dir.format(target_stationarity)):
        os.makedirs(save_dir.format(target_stationarity))

    wavfile.write(save_dir.format(target_stationarity)+f"{ID}_0.wav", samplerate, norm_s1.astype(np.float32))
    wavfile.write(save_dir.format(target_stationarity)+f"{ID}_1.wav", samplerate, norm_s2.astype(np.float32))

    k+=1

    #print(f"Sound {stationarity_scores[best_x]}:")
    #input()

In [ ]:
# Compute spectral contrast for each segment
spectral_contrast_features = []
for segment in segments:
    S = np.abs(librosa.stft(segment))  # Compute magnitude spectrogram
    contrast = librosa.feature.spectral_contrast(S=S, sr=sr)  # Compute spectral contrast
    spectral_contrast_features.append(np.mean(contrast, axis=1))  # Take mean across time

# Compute Euclidean distances between all pairs
num_segments = len(spectral_contrast_features)
max_distance = -1
best_pair = (0, 1)

for i in range(num_segments):
    for j in range(i + 1, num_segments):  # Ensure non-overlapping selection
        dist = euclidean(spectral_contrast_features[i], spectral_contrast_features[j])
        if dist > max_distance:
            max_distance = dist
            best_pair = (i, j)

# Extract the two most different segments
segment_1 = segments[best_pair[0]]
segment_2 = segments[best_pair[1]]

# Print results
print(f"Most different segments: {segment_onsets_relative_to_beginning[best_pair[0]]} and {segment_onsets_relative_to_beginning[best_pair[1]]} with distance {max_distance}")

print("best segments")
display(ipd.Audio(segment_1, rate=samplerate))
display(ipd.Audio(segment_2, rate=samplerate))


In [ ]:
# is the code doing what it is supposed to be doing?
# check other pairs
print(best_pair)
for i in range(num_segments):
    for j in range(i, num_segments):
        if i == j:
            continue
        if i == best_pair[0] and j == best_pair[1]:
            continue
        other_segment_1 = segments[i]
        other_segment_2 = segments[j]
        print(f"segment {i} and {j}")
        display(ipd.Audio(other_segment_1, rate=samplerate))
        display(ipd.Audio(other_segment_2, rate=samplerate))

# what are some selection criteria for screening for discriminable stimuli (so that I don't have to listen to all of them)?

# 1)
# - screen each 10 sec clip for the 2 sec seg that has the stationarity score most similar to -0.6. 
# - get that clip. take the first 0.5s, then take the adjacent 0.5s. (or first 0.5 or some other 0.5)
# - maybe look for the most spectrally different section

# 2) 
# - do same as 1 but chose sound with the lowest stationarity score (or try different scores like -0.5)